# chessvision — app design walkthrough

This notebook is a **guided tour of the live pipeline** so you can read and learn
the design by running it. The companion prose is in
[docs/architecture.md](../docs/architecture.md); this notebook makes each stage
*concrete* by calling the **real package functions** on a sample frame.

> It imports straight from `chessvision.*`, so what you run here is exactly what
> the app runs — nothing is reimplemented for the notebook. If a stage looks
> wrong, the bug is in the package, not in a notebook copy.

**What the app does:** a phone streams an overhead view of a chess board over
MJPEG; for every frame the pipeline finds the board, perspective-warps it to a
top-down view, works out which move was played, and records the game to PGN.

**The per-frame pipeline** (loop lives in
[chessvision/app/detect.py](../chessvision/app/detect.py)):

1. **Find the board** — `core.board.find_corners` (OpenCV chessboard corners)
2. **Outer quad** — `core.board.board_quad_from_corners`
3. **Warp** — `core.board.warp_board` → top-down 800×800 view
4. **Square mapping** — `app.detect.square_name` (orientation as a *coordinate*
   rotation, the image is left as-is)
5. **Detect** — either `core.occupancy` (model-free image diff, the default) or a
   YOLO piece model
6. **Track** — `core.tracking.GameTracker` turns flickery per-frame reads into
   legal moves and writes the PGN / FEN log

## How to use this notebook

The live app reads frames from a camera stream (`settings.stream_url`). A
notebook has no camera, so we drive the same functions with a **static sample
image** (`samples/canvas.png` — the one sample with a clean, fully visible
board) plus a couple of **synthetic** inputs for the diff and tracking stages.

Run the cells top to bottom. Each stage prints/plots its output so you can see
the data shape change as it flows through the pipeline.

In [ ]:
# --- Setup: run from the repo root so relative paths (samples/, models/) work ---
import os
from pathlib import Path

# The notebook lives in jupyter_notebooks/; walk up to the repo root.
root = Path.cwd()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
print("working dir:", os.getcwd())

import cv2
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

# The real pipeline pieces — these are the exact symbols the app uses.
from chessvision.settings import settings
from chessvision.core.board import find_corners, board_quad_from_corners, warp_board, INNER_CORNERS
from chessvision.core.occupancy import square_change_scores, changed_squares
from chessvision.core.tracking import GameTracker, label_to_symbol
from chessvision.app.detect import square_name, rotate_cell, detections_to_board

import chess


# Display an OpenCV (BGR) image inline; pass a single-channel array for masks.
def show(img_bgr, title=None, figsize=(6, 6), cmap=None):
    plt.figure(figsize=figsize)
    if img_bgr.ndim == 2:
        plt.imshow(img_bgr, cmap=cmap or "gray")
    else:
        plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    if title:
        plt.title(title)
    plt.axis("off")
    plt.show()


print("settings:",
      f"detection_mode={settings.detection_mode}",
      f"board_rotation={settings.board_rotation}",
      f"warp_padding={settings.warp_padding}",
      f"display_size={settings.display_size}")

## The module map

```
chessvision/
├── settings.py            pydantic-settings config (env prefix GRANDMASTER_)
├── core/                  shared, importable library (no GUI loops)
│   ├── board.py           find the board + perspective-warp it
│   ├── occupancy.py       model-free per-square change scores (image diff)
│   ├── tracking.py        GameTracker: stable states → legal moves → PGN/FEN
│   └── display.py         upscale-for-viewing helper
├── app/                   interactive applications
│   ├── detect.py          the main app loop (gm-detect)
│   └── view_camera.py     raw stream preview (gm-view)
└── training/              dataset + model tooling (gm-capture/-autolabel/-train/-export)
```

**Dependencies flow one way:** `app` and `training` import `core`; `core`
imports only `settings`. Nothing in `core` imports `app`. That layering is the
single most important design fact — `core` is a pure library you can call from a
notebook (which is exactly what we're doing), a test, or the app, with no GUI or
camera attached.

## Stage 1 — Find the board

`find_corners(gray)` locates the **7×7 grid of internal corners** of an 8×8
board using `cv2.findChessboardCornersSB` (a newer, more robust detector than
the classic one), falling back to the classic detector + sub-pixel refinement.
It returns the corner array, or `None` if no board is found.

In [ ]:
frame = cv2.imread("samples/canvas.png")
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

corners = find_corners(gray)
assert corners is not None, "no board found in the sample (unexpected)"
print(f"found {len(corners)} inner corners (expected {INNER_CORNERS[0]*INNER_CORNERS[1]})")

vis = frame.copy()
cv2.drawChessboardCorners(vis, INNER_CORNERS, corners, True)
show(vis, "Stage 1: 7×7 internal corners")

## Stage 2 — Extrapolate the outer quad

The detected corners only cover the **internal** intersections — they stop one
square short of the board edge on every side. `board_quad_from_corners` estimates
a one-square step vector from the grid and extrapolates outward to the four
actual board corners (returned as `[top-left, top-right, bottom-right,
bottom-left]`).

In [ ]:
quad = board_quad_from_corners(corners)
print("outer quad (tl, tr, br, bl):")
print(quad.astype(int))

vis = frame.copy()
cv2.polylines(vis, [quad.astype(int).reshape(-1, 1, 2)], True, (0, 0, 255), 3)
for label, (x, y) in zip(["TL", "TR", "BR", "BL"], quad.astype(int)):
    cv2.circle(vis, (x, y), 8, (0, 255, 0), -1)
    cv2.putText(vis, label, (x + 8, y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
show(vis, "Stage 2: extrapolated outer board quad (red), corners (green)")

## Stage 3 — Warp to a top-down view

`warp_board` perspective-warps the quad to a square top-down image
(`size`×`size`, default 800). `padding` keeps a margin of original context around
the edges so tall pieces overhanging the boundary stay visible. With the default
`warp_padding=40` the output is `880×880`: a centred `800×800` board inside a
40px frame.

In [ ]:
warped = warp_board(frame, quad, size=settings.display_size, padding=settings.warp_padding)
print("warped shape:", warped.shape, "(board area is 800×800 inside a 40px pad)")
show(warped, "Stage 3: top-down warped board (with 40px padding)")

## Stage 4 — Square mapping & orientation

This is the most subtle design choice. The camera sees the board at some
arbitrary rotation, but **the app never rotates the displayed image** — it leaves
the view in the raw-feed orientation and instead rotates the *coordinate → square
name* mapping. `square_name(col, row)` applies `settings.board_rotation`
(a `rotate_cell` index rotation) and `flip_orientation`, so the picture stays
put while squares still read correctly.

The **live rig's** default is `board_rotation=90` → **a1 bottom-right** (see
[docs/configuration.md](../docs/configuration.md#board-orientation)). But
`samples/canvas.png` is an **older capture shot from a different angle, with a1
at the top-left**, so the default mapping would mislabel it. We fix it exactly
the way you would live — by changing `settings.board_rotation` (pressing **`o`**
cycles it 90° at runtime; here we just set it directly). For this sample **`270`**
is the value that puts **a1 in the top-left**:

- top-left → `a1`, top-right → `a8`, bottom-left → `h1`, bottom-right → `h8`
- the rank-1 (white) pieces run down the left column — matching the Stage 3 warp.

Because `square_name` reads `settings` live, setting it here also corrects the
changed-square names in Stage 5. Below we label every cell with the name
`square_name` assigns it; the red corner squares show the mapping.

In [ ]:
# This sample (canvas.png) is an older capture with a1 at the TOP-LEFT, unlike
# the live rig's default (board_rotation=90 -> a1 bottom-right). We adjust the
# mapping for THIS image exactly as the 'o' key does at runtime — it only mutates
# settings.board_rotation, which square_name reads live. (Set back to 90 to see
# how the live default would mislabel this older capture.)
settings.board_rotation = 270      # a1 top-left for this sample
settings.flip_orientation = False

labelled = warped.copy()
pad = settings.warp_padding
cell = settings.display_size // 8  # 100 px per square inside the padded area

for col in range(8):
    for row in range(8):
        name = square_name(col, row)
        cx = pad + int((col + 0.5) * cell)
        cy = pad + int((row + 0.5) * cell)
        cv2.rectangle(labelled, (pad + col * cell, pad + row * cell),
                      (pad + (col + 1) * cell, pad + (row + 1) * cell), (0, 180, 0), 1)
        color = (0, 0, 255) if name in {"a1", "h1", "a8", "h8"} else (0, 220, 0)
        cv2.putText(labelled, name, (cx - 22, cy + 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2, cv2.LINE_AA)

show(labelled, f"Stage 4: square names with board_rotation={settings.board_rotation}", figsize=(7, 7))
print("note how rotate_cell remaps grid (col,row) -> rotated index before naming:")
for (c, r) in [(0, 0), (7, 0), (0, 7), (7, 7)]:
    print(f"  grid cell (col={c}, row={r})  ->  rotated {rotate_cell(c, r, settings.board_rotation)}  ->  {square_name(c, r)}")

## Stage 5 — Detection, mode A: model-free image **diff** (the default)

Identifying *which* piece sits on a square is hard; `diff` mode sidesteps it
entirely. It keeps a **reference image** of the board as of the last committed
move and asks only *which squares changed*. `core.occupancy`:

- `square_change_scores(warped, reference)` → an **8×8 float array** of the mean
  absolute grayscale difference inside the central `inner` fraction of each
  square (sampling only the centre avoids grid lines and tall leaning
  neighbours).
- `changed_squares(scores, threshold)` → the set of `(col, row)` cells above
  `settings.diff_change_threshold`.

There's no second board photo in `samples/`, so we **synthesize a move**: copy
the warped board as the reference, then draw two blobs on a copy to mimic a piece
leaving one square and arriving on another. The real `square_change_scores`
then scores the change.

In [ ]:
reference = warped.copy()
current = warped.copy()

def cell_center(col, row):
    return pad + int((col + 0.5) * cell), pad + int((row + 0.5) * cell)

# Mimic a piece leaving (col=4,row=6) and arriving at (col=4,row=4).
cv2.circle(current, cell_center(4, 6), 30, (235, 235, 235), -1)  # square emptied -> bg colour
cv2.circle(current, cell_center(4, 4), 30, (25, 25, 25), -1)     # piece appears -> dark blob

scores = square_change_scores(current, reference,
                              padding=settings.warp_padding,
                              inner=settings.diff_inner_fraction)
changed = changed_squares(scores, settings.diff_change_threshold)

print("threshold:", settings.diff_change_threshold, " max score:", round(float(scores.max()), 1))
print("changed (col,row) cells:", sorted(changed))
print("changed square names:", sorted(square_name(c, r) for c, r in changed))

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(cv2.cvtColor(current, cv2.COLOR_BGR2RGB)); ax[0].set_title("synthetic 'current' frame"); ax[0].axis("off")
im = ax[1].imshow(scores, cmap="hot"); ax[1].set_title("8×8 change scores"); ax[1].set_xlabel("col"); ax[1].set_ylabel("row")
fig.colorbar(im, ax=ax[1], fraction=0.046)
plt.show()

Note the change set is reported as `(col, row)` grid cells; `square_name` turns
them into chess squares using the same orientation mapping from Stage 4. In the
real loop ([detect.py](../chessvision/app/detect.py)), a frame with more than
`settings.diff_max_changed` changed cells (e.g. a hand crossing the board) is
discarded as noise before it ever reaches the tracker.

## Stage 5 — Detection, mode B: the YOLO piece **model** (optional)

`model` mode runs a YOLO detector (12 classes: 6 piece types × 2 colours) on the
warped view, and `detections_to_board` assigns each detection to a square using
the **bottom-centre** of its box (a piece's base sits on its square, not the box
centre), keeping the highest-confidence detection per square.

⚠️ The bundled model is still under-trained, so its reads on this synthetic
rendered board are **unreliable** — it tends to hallucinate plenty of
high-confidence but wrong piece labels. That unreliability is exactly *why*
`diff` is the default mode. This cell shows the call shape and how
`detections_to_board` + `label_to_symbol` turn boxes into squares/symbols; don't
trust the labels themselves here. It's wrapped so a poor result can't stop the
notebook.

In [ ]:
try:
    from ultralytics import YOLO
    model = YOLO(str(settings.pieces_model_path), task="detect")
    results = model(warped, verbose=False, conf=settings.pieces_conf_threshold)[0]
    board = detections_to_board(results, warped.shape[1], warped.shape[0],
                                padding=settings.warp_padding)
    print(f"model detected {len(results.boxes)} boxes -> {len(board)} occupied squares")
    for sq, (label, conf) in sorted(board.items()):
        print(f"  {sq}: {label} ({conf:.2f}) -> python-chess symbol {label_to_symbol(label)}")
    if len(results.boxes):
        show(results.plot(), "Stage 5B: YOLO detections on the warped board")
    else:
        print("(no detections — model under-trained on this sample; use diff mode)")
except Exception as e:
    print(f"model step skipped: {type(e).__name__}: {e}")

## Stage 6 — `GameTracker`: turning flickery reads into a game

This is the brain of the app ([core/tracking.py](../chessvision/core/tracking.py)).
Per-frame detections flicker, so the tracker:

1. **Debounces** — a state must be seen unchanged for `stability_frames`
   consecutive frames before it's trusted.
2. **Snaps to a legal move** — it asks **python-chess** for the legal move that
   best explains the observation. In `diff` mode (`update_changed`) it picks the
   legal move whose *own* altered squares match the changed-square set; in
   `model` mode (`update`) it picks the legal move whose *resulting position*
   best matches the detected pieces. Either way, captures, castling, en passant
   and promotion **fall out of the rules for free**, and detections matching no
   legal move are ignored.
3. **Records** — after every accepted move it rewrites the `.pgn` and a
   human-readable `.fen.log`, so both always reflect the current game.

We'll drive it **purely in memory** (no images): for each move we compute the
squares it alters and feed that set `stability_frames` times, mimicking a stable
diff read.

In [ ]:
import tempfile

tmp = Path(tempfile.mkdtemp())
tracker = GameTracker(tmp / "game.pgn", tmp / "game.fen.log",
                      stability_frames=settings.game_stability_frames)


# The square names a move changes — what an image diff would light up.
def altered_names(board: chess.Board, uci: str) -> set[str]:
    move = chess.Move.from_uci(uci)
    before = board.piece_map()
    board.push(move); after = board.piece_map(); board.pop()
    return {chess.square_name(s) for s in set(before) | set(after)
            if before.get(s) != after.get(s)}


# Play a short opening. We feed each move's changed-square set until it's stable.
for uci in ["e2e4", "e7e5", "g1f3", "b8c6", "f1b5"]:  # Ruy Lopez
    changed = altered_names(tracker.board, uci)
    for _ in range(settings.game_stability_frames):
        tracker.update_changed(changed)

print("\nfinal position:")
print(tracker.board)
print("\nmoves recorded:", tracker.move_count, "| last:", tracker.last_san)

Watch the `[rec] ...` lines above: each is a move the tracker *inferred from the
changed squares alone* — it never saw which piece moved, only that those squares
changed, yet it recovered the full notation. Now the files it wrote:

In [ ]:
print("=== game.pgn ===")
print((tmp / "game.pgn").read_text())
print("=== game.fen.log ===")
print((tmp / "game.fen.log").read_text())

## Putting it together — the real loop

In [app/detect.py](../chessvision/app/detect.py) the `main()` loop wires these
exact stages together for every camera frame:

```
read frame ─▶ find_corners ─▶ board_quad_from_corners ─▶ warp_board
                                                            │
                          ┌─────────────────────────────────┤
                  diff mode│                                 │model mode
        square_change_scores│                                 │YOLO(warped)
           changed_squares  │                                 │detections_to_board
                            ▼                                 ▼
              tracker.update_changed(names) ───┐   ┌─── tracker.update(board)
                                               ▼   ▼
                                          GameTracker  ─▶  games/*.pgn + *.fen.log
```

Two design tricks make it robust on real hardware:

- **Lock the board (`k`)** — a full set of pieces hides the inner grid that
  `find_corners` needs, so the app caches the warp once on the empty board and
  reuses it every frame. `diff` recording auto-locks on `r`.
- **Coordinate-only orientation** — the displayed image always matches the raw
  feed; only `square_name` rotates (Stage 4), so what you see and what's recorded
  never disagree.

### Where to read next

| To understand… | Read |
| --- | --- |
| The full per-frame loop & keybindings | [app/detect.py](../chessvision/app/detect.py) |
| Board geometry | [core/board.py](../chessvision/core/board.py) |
| The model-free diff | [core/occupancy.py](../chessvision/core/occupancy.py) |
| Move inference & recording | [core/tracking.py](../chessvision/core/tracking.py) |
| Every setting | [docs/configuration.md](../docs/configuration.md) + [settings.py](../chessvision/settings.py) |
| The big picture | [docs/architecture.md](../docs/architecture.md) |

To run the real thing against a camera: `uv run gm-view` (preview) then
`uv run gm-detect` (full pipeline).